# 02 - Preprocessing

In [2]:
import pandas as pd

In [8]:
# Load data 
months = ["202512", "202601", "202602", "202603", "202604", "202605"]
files = [f"california/CRMLSSold{m}.csv" for m in months]

df = pd.concat([pd.read_csv(f, low_memory=False) for f in files], ignore_index=True)
df = df[(df["PropertyType"] == "Residential") & (df["PropertySubType"] == "SingleFamilyResidence")]
df.shape

(61727, 78)

## Select columns

In [9]:
num_cols = ["LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "LotSizeSquareFeet", "YearBuilt",
            "GarageSpaces", "ParkingTotal", "AssociationFee", "Latitude", "Longitude"]
bool_cols = ["PoolPrivateYN", "ViewYN", "FireplaceYN", "NewConstructionYN", "AttachedGarageYN"]
cat_cols = ["CountyOrParish"]

# Drop listings that show up in more than one file
df = df.drop_duplicates("ListingKey")
df["CloseMonth"] = pd.to_datetime(df["CloseDate"]).dt.strftime("%Y%m")
df = df[["ClosePrice", "CloseMonth"] + num_cols + bool_cols + cat_cols]
df.isna().sum()

ClosePrice                   0
CloseMonth                   0
LivingArea                  30
BedroomsTotal                0
BathroomsTotalInteger        1
LotSizeSquareFeet         1081
YearBuilt                   36
GarageSpaces              2353
ParkingTotal                 0
AssociationFee           17614
Latitude                     9
Longitude                    9
PoolPrivateYN             4833
ViewYN                    5295
FireplaceYN                 58
NewConstructionYN         4685
AttachedGarageYN          7521
CountyOrParish               0
dtype: int64

## Missing values

In [11]:
# Few rows are missing these, so drop them
df = df.dropna(subset=["LivingArea", "BathroomsTotalInteger", "LotSizeSquareFeet", "YearBuilt", "Latitude"])

# No fee listed means no HOA
df["AssociationFee"] = df["AssociationFee"].fillna(0)

# Fill the amount of garage spaces with the median amount of garage spaces
df["GarageSpaces"] = df["GarageSpaces"].fillna(df["GarageSpaces"].median())

# Treat missing Yes/No as No
df[bool_cols] = df[bool_cols].eq(True).astype(int)

df.isna().sum()

ClosePrice               0
CloseMonth               0
LivingArea               0
BedroomsTotal            0
BathroomsTotalInteger    0
LotSizeSquareFeet        0
YearBuilt                0
GarageSpaces             0
ParkingTotal             0
AssociationFee           0
Latitude                 0
Longitude                0
PoolPrivateYN            0
ViewYN                   0
FireplaceYN              0
NewConstructionYN        0
AttachedGarageYN         0
CountyOrParish           0
dtype: int64

## Encoding

In [12]:
df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)
df.shape

(60542, 74)

In [ ]:
# Save the data to csv
df.to_csv("cleaned.csv", index=False)

## Train/test split

In [ ]:
# Test on the most recent month, train on the X months before it
X = 4
train = df[df["CloseMonth"].isin(months[-1 - X:-1])].copy()
test = df[df["CloseMonth"] == months[-1]].copy()
train.shape, test.shape

## Normalize

In [ ]:
# Normalize numeric features using the training set's mean and std
mean, std = train[num_cols].mean(), train[num_cols].std()
train[num_cols] = (train[num_cols] - mean) / std
test[num_cols] = (test[num_cols] - mean) / std
train[num_cols].describe()